# Figure S9: signature scores adjusted for baseline clinical variables

Forest plots of the MICE-pooled logistic regressions of response (R vs NR) on the LN, FMAC and TEX
scores (standardized, so odds ratios are per SD). The models are fitted in
`code/01_compute/04_clinical_adjustment_mice.ipynb` with the five baseline variables of the Methods
(IPI, LDH, Day -30 LDH, bridging therapy, stage); the same fits give Tables S3 and S4.

**Inputs** (`results/mice/`)
- `logistic_by_adjustment.csv`: score OR unadjusted, adjusted for each variable, and fully adjusted
- `univariate_logistic.csv`: univariate OR of each score and each clinical variable

**Outputs** (`plots/`)
- `SFig9A_forest_logistic_by_adjustment.pdf`
- `SFig9B_forest_logistic_univariate.pdf`

In [ ]:
source("../../utils/setup.R")

## Inputs

In [ ]:
SIGNATURES <- c("LN_set1", "FMAC_set1", "TEX_set1")

logistic_adj  <- read.csv(res_in("mice", "logistic_by_adjustment.csv"))
logistic_univ <- read.csv(res_in("mice", "univariate_logistic.csv"))

logistic_adj %>% select(signature, adjustment, n, OR, OR_lo, OR_hi, p_value)

## Figure S9A — Odds ratio per SD by adjustment, LN / FMAC / TEX

In [ ]:
# Rows top to bottom: Unadjusted, "+ <variable>" in model order, fully adjusted (file order)
ADJ_ORDER <- unique(logistic_adj$adjustment)

p_sfig9a <- logistic_adj %>%
  mutate(adjustment = factor(adjustment, levels = rev(ADJ_ORDER)),
         signature  = factor(signature, levels = SIGNATURES)) %>%
  filter(!is.na(OR)) %>%
  ggplot(aes(x = adjustment, y = OR, ymin = OR_lo, ymax = OR_hi, color = signature)) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey60", linewidth = 0.4) +
  geom_pointrange(linewidth = 0.5, size = 0.2) +
  scale_color_manual(values = SIGNATURE_COLORS, guide = "none") +
  scale_y_log10() +
  coord_flip() +
  facet_wrap(~signature, nrow = 1,
             labeller = as_labeller(c(LN_set1 = "LN Score", FMAC_set1 = "FMAC Score", TEX_set1 = "TEX Score"))) +
  labs(x = NULL, y = "Odds Ratio per SD increase") +
  theme_cfrna_print() +
  theme_axis_lines() +
  theme(strip.text = element_text(family = "Helvetica", size = 8, color = "black", margin = margin(1, 0, 1, 0)),
        strip.background = element_rect(fill = NA, color = "black"))

save_plot(p_sfig9a, "SFig9A_forest_logistic_by_adjustment", width = 4.5, height = 2)
p_sfig9a

## Figure S9B — Univariate odds ratios, scores (blue) vs clinical variables (gray)

In [ ]:
logistic_univ %>% select(label, n, OR, OR_lo, OR_hi, p_value)

p_sfig9b <- logistic_univ %>%
  mutate(label = factor(label, levels = rev(label)),
         group = if_else(variable %in% SIGNATURES, "Signature score", "Baseline covariate")) %>%
  ggplot(aes(x = label, y = OR, ymin = OR_lo, ymax = OR_hi, color = group)) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey60", linewidth = 0.4) +
  geom_pointrange(linewidth = 0.5, size = 0.2) +
  scale_color_manual(values = UNIVARIATE_GROUP_COLORS) +
  scale_y_log10() +
  coord_flip() +
  labs(x = NULL, y = "Odds Ratio", color = NULL) +
  theme_cfrna_print() +
  theme_axis_lines()

save_plot(p_sfig9b, "SFig9B_forest_logistic_univariate", width = 2.5, height = 2)
p_sfig9b